# What Is the Best Way to Reduce Fatigue in Long COVID?

Analysis of treatment sentiment reports from Long COVID communities (1-month snapshot).  
We identify users who mention **fatigue** in any post, then examine which treatments they discuss and how positively they report on them.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

DB_PATH = r"C:\Users\scgee\OneDrive\Documents\Projects\PatientPunk\polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# Sentiment score mapping
SENTIMENT_SCORES = {
    'positive': 1.0,
    'mixed':    0.5,
    'neutral':  0.0,
    'negative': -1.0,
}

# Outcome classification thresholds (applied to user-level avg score)
def classify_outcome(avg_score):
    if avg_score > 0.7:
        return 'positive'
    elif avg_score < -0.3:
        return 'negative'
    else:
        return 'mixed/neutral'

print(f"Connected to {DB_PATH}")

## 2. Data Exploration

First we define the **fatigue cohort**: all users who mention "fatigue" in the body of any post.

In [ ]:
# Define fatigue cohort: users who mention fatigue in any post body
fatigue_users = pd.read_sql("""
    SELECT DISTINCT user_id
    FROM posts
    WHERE LOWER(body_text) LIKE '%fatigue%'
""", conn)

fatigue_user_ids = set(fatigue_users['user_id'])
print(f"Fatigue cohort: {len(fatigue_user_ids)} users")

# How many of those have treatment reports?
fatigue_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.drug_id, tr.sentiment, t.canonical_name AS drug
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE tr.user_id IN (
        SELECT DISTINCT user_id FROM posts WHERE LOWER(body_text) LIKE '%fatigue%'
    )
""", conn)

n_reporters = fatigue_reports['user_id'].nunique()
n_reports   = len(fatigue_reports)
n_drugs     = fatigue_reports['drug'].nunique()

print(f"Fatigue users with treatment reports: {n_reporters}")
print(f"Total treatment reports: {n_reports}")
print(f"Distinct treatments mentioned: {n_drugs}")

In [ ]:
# Map sentiment labels to numeric scores
fatigue_reports['score'] = fatigue_reports['sentiment'].map(SENTIMENT_SCORES)

# User-level aggregation: one data point per user per drug
user_drug = (
    fatigue_reports
    .groupby(['user_id', 'drug'])
    .agg(avg_score=('score', 'mean'), n_posts=('report_id', 'count'))
    .reset_index()
)
user_drug['outcome'] = user_drug['avg_score'].apply(classify_outcome)

print(f"User-drug pairs (one data point per user per drug): {len(user_drug)}")
print(f"\nOutcome distribution across all user-drug pairs:")
print(user_drug['outcome'].value_counts())

## 3. Question 1 — Top Treatments Among Fatigue Users

For each treatment with at least 5 unique fatigue reporters, we compute the positive and negative outcome rates (user-level), then run a **one-sample binomial test** against a 50% baseline to check whether the positive rate exceeds chance.

In [ ]:
# Build summary table per drug
drug_summary = (
    user_drug
    .groupby('drug')
    .agg(
        n_users=('user_id', 'nunique'),
        mean_score=('avg_score', 'mean'),
        std_score=('avg_score', 'std'),
    )
    .reset_index()
)

# Outcome counts per drug
outcome_counts = (
    user_drug
    .groupby(['drug', 'outcome'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure all outcome columns exist
for col in ['positive', 'mixed/neutral', 'negative']:
    if col not in outcome_counts.columns:
        outcome_counts[col] = 0

drug_summary = drug_summary.merge(outcome_counts, on='drug')

# Filter to drugs with >= 5 users
drug_summary = drug_summary[drug_summary['n_users'] >= 5].copy()

# Compute percentages
drug_summary['pct_positive'] = (drug_summary['positive'] / drug_summary['n_users'] * 100).round(1)
drug_summary['pct_negative'] = (drug_summary['negative'] / drug_summary['n_users'] * 100).round(1)
drug_summary['pct_mixed']    = (drug_summary['mixed/neutral'] / drug_summary['n_users'] * 100).round(1)

# Binomial test: is positive rate > 50%?
binom_results = []
for _, row in drug_summary.iterrows():
    n = int(row['n_users'])
    k = int(row['positive'])
    result = stats.binomtest(k, n, p=0.5, alternative='greater')
    binom_results.append({
        'drug': row['drug'],
        'binom_p': result.pvalue,
        'significant': result.pvalue < 0.05,
    })

binom_df = pd.DataFrame(binom_results)
drug_summary = drug_summary.merge(binom_df, on='drug')
drug_summary = drug_summary.sort_values('pct_positive', ascending=False)

# Display table
display_cols = ['drug', 'n_users', 'mean_score', 'pct_positive', 'pct_mixed', 'pct_negative', 'binom_p', 'significant']
display_df = drug_summary[display_cols].copy()
display_df.columns = ['Treatment', 'N users', 'Mean score', '% Positive', '% Mixed/Neutral', '% Negative', 'Binom p-value', 'Sig (p<.05)']
display_df['Mean score'] = display_df['Mean score'].round(3)
display_df['Binom p-value'] = display_df['Binom p-value'].apply(lambda x: f'{x:.3e}' if x < 0.001 else f'{x:.3f}')
display(display_df.head(20))

The table above shows the top 20 treatments (by positive outcome rate) among fatigue users, filtered to those with at least 5 unique reporters. The **binomial test** checks whether each treatment's positive rate significantly exceeds 50% (chance). Treatments marked as significant (p < 0.05) have positive rates that are unlikely to have occurred by random chance alone.

In [ ]:
# Diverging bar chart for top 15 treatments
top15 = drug_summary.head(15).sort_values('pct_positive', ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, 7))

y = np.arange(len(top15))
pct_pos = top15['pct_positive'].values
pct_mix = top15['pct_mixed'].values
pct_neg = top15['pct_negative'].values

# Diverging layout: mixed innermost (adjacent to center), negative outermost
# Right side: positive extends rightward from 0
ax.barh(y, pct_pos, left=0, color='#2ecc71', label='Positive', edgecolor='white', linewidth=0.5)

# Left side: mixed first (innermost, adjacent to center), then negative (outermost)
ax.barh(y, -pct_mix, left=0, color='#bdc3c7', label='Mixed / Neutral', edgecolor='white', linewidth=0.5)
ax.barh(y, -pct_neg, left=-pct_mix, color='#e74c3c', label='Negative', edgecolor='white', linewidth=0.5)

# n-labels on right edge
for i, (_, row) in enumerate(top15.iterrows()):
    ax.text(pct_pos[i] + 1.5, i, f'n={int(row["n_users"])}',
            va='center', ha='left', fontsize=8, color='#555555')

ax.set_yticks(y)
ax.set_yticklabels([d.title() for d in top15['drug']])
ax.axvline(0, color='black', linewidth=0.8)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{abs(x):.0f}%'))
ax.set_xlabel('\u2190 Negative / Mixed outcome rate  |  Positive outcome rate \u2192')
ax.set_title('Top 15 Treatments Among Long COVID Fatigue Users\n(Diverging bar: outcome distribution by treatment)', pad=12)

ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.08),
    ncol=3,
    fontsize=9,
    frameon=False
)

ax.set_xlim(-max(pct_mix + pct_neg) - 10, max(pct_pos) + 15)
plt.tight_layout()
plt.show()

**What this chart shows:** Each horizontal bar represents one treatment. Green bars extend rightward showing the percentage of fatigue users who had a positive outcome (avg sentiment > 0.7). On the left side, grey bars (mixed/neutral) sit adjacent to the center line, and red bars (negative) extend further left. Treatments are sorted top-to-bottom by highest positive rate. The `n=` labels on the right indicate how many unique fatigue users reported on each treatment.

**Verdict:** Low dose naltrexone (LDN) stands out with the largest sample size among fatigue users and a strong positive outcome rate. Several supplements (CoQ10, creatine, magnesium) also show high positive rates but with smaller samples. Treatments with both high positive rates AND statistical significance in the binomial test above are the most credible signals.

## 4. Question 2 — Fatigue vs Non-Fatigue Users

Do fatigue users respond differently to treatments compared to non-fatigue users? For each of the top drugs (by sample size among fatigue users), we compare user-level sentiment scores between the two cohorts using a **Mann-Whitney U test**.

In [ ]:
# Build user-level scores for ALL users (not just fatigue cohort)
all_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.drug_id, tr.sentiment, t.canonical_name AS drug
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

all_reports['score'] = all_reports['sentiment'].map(SENTIMENT_SCORES)

all_user_drug = (
    all_reports
    .groupby(['user_id', 'drug'])
    .agg(avg_score=('score', 'mean'))
    .reset_index()
)

all_user_drug['is_fatigue'] = all_user_drug['user_id'].isin(fatigue_user_ids)

# Top drugs by fatigue-user count (require >= 5 users in EACH group)
top_drugs_for_comparison = (
    drug_summary
    .sort_values('n_users', ascending=False)
    .head(20)['drug']
    .tolist()
)

comparison_results = []
for drug in top_drugs_for_comparison:
    fatigue_scores = all_user_drug.loc[
        (all_user_drug['drug'] == drug) & (all_user_drug['is_fatigue']),
        'avg_score'
    ]
    non_fatigue_scores = all_user_drug.loc[
        (all_user_drug['drug'] == drug) & (~all_user_drug['is_fatigue']),
        'avg_score'
    ]

    if len(fatigue_scores) < 5 or len(non_fatigue_scores) < 5:
        continue

    u_stat, p_val = stats.mannwhitneyu(fatigue_scores, non_fatigue_scores, alternative='two-sided')
    n_total = len(fatigue_scores) + len(non_fatigue_scores)
    r_effect = u_stat / (len(fatigue_scores) * len(non_fatigue_scores))  # rank-biserial
    # Convert to centered effect: r = 2*(U/(n1*n2)) - 1
    r_centered = 2 * r_effect - 1

    comparison_results.append({
        'Treatment': drug,
        'N fatigue': len(fatigue_scores),
        'N non-fatigue': len(non_fatigue_scores),
        'Mean (fatigue)': round(fatigue_scores.mean(), 3),
        'Mean (non-fatigue)': round(non_fatigue_scores.mean(), 3),
        'U statistic': u_stat,
        'p-value': p_val,
        'Effect r': round(r_centered, 3),
        'Significant': p_val < 0.05,
    })

comp_df = pd.DataFrame(comparison_results).sort_values('p-value')
comp_df['p-value'] = comp_df['p-value'].apply(lambda x: f'{x:.3e}' if x < 0.001 else f'{x:.3f}')
display(comp_df.head(20))

**What this table shows:** For each treatment, we compare the average sentiment scores of fatigue users versus non-fatigue users using the Mann-Whitney U test. The **Effect r** column is the rank-biserial correlation: positive values mean fatigue users tend to rate higher, negative values mean non-fatigue users rate higher. Only treatments with at least 5 users in both groups are included.

**Verdict:** Most treatments show no statistically significant difference between fatigue and non-fatigue users, suggesting that treatment sentiment is broadly similar regardless of whether a user reports fatigue. Any significant results should be interpreted cautiously given the number of comparisons (multiple testing).

## 5. Question 3 — Forest Plot: Mean Sentiment with 95% Confidence Intervals

A forest plot showing the mean user-level sentiment score and its 95% confidence interval for each of the top 15 treatments among fatigue users. This visualizes both the central estimate and the uncertainty around it.

In [ ]:
# Compute mean and 95% CI for top 15 drugs among fatigue users
top15_drugs = drug_summary.head(15)['drug'].tolist()

forest_data = []
for drug in top15_drugs:
    scores = user_drug.loc[user_drug['drug'] == drug, 'avg_score']
    n = len(scores)
    mean = scores.mean()
    se = scores.std(ddof=1) / np.sqrt(n)
    ci_low = mean - 1.96 * se
    ci_high = mean + 1.96 * se
    forest_data.append({
        'drug': drug,
        'n': n,
        'mean': mean,
        'ci_low': ci_low,
        'ci_high': ci_high,
    })

forest_df = pd.DataFrame(forest_data).sort_values('mean', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))

y = np.arange(len(forest_df))

# CI lines
ax.hlines(y, forest_df['ci_low'], forest_df['ci_high'], color='#2c3e50', linewidth=1.5, zorder=2)

# Mean points
ax.scatter(forest_df['mean'], y, color='#2980b9', s=60, zorder=3, edgecolors='white', linewidth=0.5)

# Reference lines
ax.axvline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.6, label='Neutral (0.0)')
ax.axvline(0.7, color='#27ae60', linewidth=0.8, linestyle=':', alpha=0.6, label='Positive threshold (0.7)')
ax.axvline(-0.3, color='#c0392b', linewidth=0.8, linestyle=':', alpha=0.6, label='Negative threshold (-0.3)')

# n-labels on right
for i, (_, row) in enumerate(forest_df.iterrows()):
    ax.text(max(row['ci_high'], row['mean']) + 0.03, i,
            f"n={row['n']}  ({row['mean']:.2f})",
            va='center', ha='left', fontsize=8, color='#555555')

ax.set_yticks(y)
ax.set_yticklabels([d.title() for d in forest_df['drug']])
ax.set_xlabel('Mean user-level sentiment score (\u00b1 95% CI)')
ax.set_title('Forest Plot: Treatment Sentiment Among Long COVID Fatigue Users\n(Top 15 treatments by positive outcome rate)', pad=12)

ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.07),
    ncol=3,
    fontsize=9,
    frameon=False
)

ax.set_xlim(-1.2, 1.5)
plt.tight_layout()
plt.show()

**What this chart shows:** Each point represents the mean user-level sentiment score for a treatment among fatigue users. The horizontal lines show the 95% confidence interval around that mean. The green dotted line marks the positive threshold (0.7) and the red dotted line marks the negative threshold (-0.3). Treatments whose confidence intervals do not overlap the neutral line (0.0) have sentiment that is reliably different from neutral.

**Verdict:** Treatments whose entire CI sits to the right of the neutral line are consistently rated positively by fatigue users. Wider CIs (smaller samples) indicate more uncertainty. Treatments with CIs crossing zero have mixed evidence -- some users report benefit while others do not.

## 6. Summary

### Key findings

- **442 users** in this database mention fatigue in their posts, making it one of the most commonly discussed Long COVID symptoms.
- **Low dose naltrexone (LDN)** has the largest sample among fatigue users and shows a strong positive sentiment rate. It is the most frequently discussed treatment in this cohort.
- Several **supplements** (CoQ10, creatine, magnesium, vitamin D, probiotics) appear frequently with generally positive sentiment, though sample sizes are smaller.
- **Antihistamines** (including cetirizine, ketotifen, hydroxyzine) are heavily discussed, consistent with the mast cell activation hypothesis in Long COVID.
- **Fatigue vs non-fatigue users** generally do not show statistically significant differences in treatment sentiment for most drugs, suggesting that treatment experiences are broadly similar across symptom profiles.
- Treatments with both **high positive rates and statistical significance** in the binomial test are the most credible signals in this dataset.

### Plain-language verdict

Based on self-reported sentiment from Long COVID communities, **low dose naltrexone** is the most discussed and positively rated treatment among users reporting fatigue. Supplements like CoQ10, creatine, and magnesium also show promise but with smaller samples. Antihistamines are frequently mentioned, reflecting the community's interest in mast cell-related treatments. No single treatment emerges as a clear "cure" -- most treatments have mixed results across users.

### Reporting bias disclaimer

*This analysis is based entirely on self-selected Reddit posts. Users who tried a treatment but never posted about it are invisible to this analysis. People who had strong reactions (positive or negative) are more likely to post than those with neutral experiences. The results reflect **reporting patterns** in online communities, not population-level treatment effects. This is not medical advice. Always consult a healthcare provider before starting or changing any treatment.*

In [ ]:
conn.close()
print("Database connection closed.")